# Physical Pain
notebook containing experiments, preps & wrangling of physical pain data ([IHME](https://www.healthdata.org/sites/default/files/2025-10/GBD_2023_Booklet_Final_2025.10.17.pdf))


In [ ]:
# Imports
from typing import Dict, List, Optional, Set, Tuple
import gzip
import os
import pandas as pd

In [ ]:
# Path constants
BASE_PATH = os.path.join('..', 'data')
# datasets prior to improvements
#DATASET_PATH = os.path.join(BASE_PATH, 'raw', 'prevalence.csv')
#DS_CONVERTED_PATH1 = os.path.join(BASE_PATH, 'raw', 'prevalence1.csv')
#DS_CONVERTED_PATH2 = os.path.join(BASE_PATH, 'raw', 'prevalence2.csv') # NOTE: this file is useless as I don't know how it is encoded
# new datasets
DATASET_PATH = os.path.join(BASE_PATH, 'raw', 'physical', 'prevalence_by_pixel.csv.gz')

OUTPUT_PATH = os.path.join(BASE_PATH, "raw", "physical", "percent_filtered2.csv.gz")
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
if os.path.exists(OUTPUT_PATH):
    os.remove(OUTPUT_PATH)

In [ ]:
# Other constants
CHUNK_SIZE = 100_000  # for streaming the huge data file
COL_PREVALENCE = "pixel_abs_prevalence"

In [ ]:
# Quick sanity check
nrows = 20
pix = pd.read_csv(
    DATASET_PATH,
    skiprows=2,
    nrows=nrows,
    compression="gzip",   # <-- fix for UnicodeDecodeError
)
pix.head(n=nrows)

In [ ]:
# infer data types
sample = pd.read_csv(DATASET_PATH, nrows=10_000, compression="gzip")
print(sample.dtypes)

In [ ]:
# Check validity of "Percent" values
sample = pd.read_csv(DATASET_PATH, nrows=10_000, compression="gzip")
s_filtered = sample[sample['metric_name'] == "Percent"]
s_filtered['pixel_abs_prevalence'].max()
#s_filtered.head()

In [ ]:
# Analyse causes (treating them categorically since we don't expect many different causes)
# this list here seems to be a list of all possible causes (== target conditions?):
#    https://github.com/inescgu/pain-world/blob/main/prevalence.py#L8
# but this includes way more than the three we found in the data
causes = ['Osteoarthritis', 'Rheumatoid arthritis', 'Headache disorders']   # found by executing the code below
causes: Set[str] = set()
chunk_size = CHUNK_SIZE * 10      # bigger chunk size since we only care about one column and make it categorically
reader = pd.read_csv(
    DATASET_PATH,
    chunksize=chunk_size,
    usecols=["cause_name"],
    dtype={
      "cause_name": "category",
    },
    #compression="gzip",
)
last_size = len(causes)
i = 0
print(f"Start streaming with chunk size = {chunk_size}")
while True:
  try:
    chunk = next(reader)
    causes.update(chunk["cause_name"].cat.categories)
    cur_size = len(causes)
    if cur_size > last_size:
      print("causes = ", causes)
      last_size = cur_size
    i += 1
  except StopIteration:
    break
  except Exception as ex:
    print("#####################################################")
    print(f"Error occurred at cunk #{i}: ", ex)
    print("#####################################################")
print(f"DONE after {i} chunks")

In [ ]:
# Confirm structure of alternating absolute and relative numbers
# result: 
#   - for 1_000_000 it looks good
#   - for 10_000_000 it looks horrendus (79% between 0 & 1)
#   - for 100_000_000 it's even worse: 86% between 0 & 1
sample = pd.read_csv(DATASET_PATH, 
                     nrows=10_000_000, 
                     compression="gzip"
                     )

#print(sample[COL_PREVALENCE].describe())
#print(sample[COL_PREVALENCE].min(), sample[COL_PREVALENCE].max())

mask = sample[COL_PREVALENCE].between(0, 1, inclusive='left')
print(mask.value_counts())

relative = sample[mask]
duplicates = relative.duplicated(subset=["lat", "lon"])
if duplicates.any():
  print("Duplicates:")
  print(relative.loc[duplicates, ["lat", "lon", COL_PREVALENCE]])

In [ ]:
# check percentage of values < 1
sample = pd.read_csv(DATASET_PATH, nrows=10_000_000, compression="gzip")
mask = sample["pixel_abs_prevalence"].between(0, 1, inclusive='left')
#mask = sample["pixel_abs_upper"].between(0, 1, inclusive='left')
print(mask.value_counts())

In [ ]:
# inspect the presumably absolute entries with values <= 1
start_row = 16883+1
nrows = 7
misleading_abs = pd.read_csv(DATASET_PATH, skiprows=start_row, nrows=nrows, compression="gzip")
misleading_abs.head(nrows)

In [ ]:
# Create a stripped down dataset with only the needed columns and rows
chunk_size = CHUNK_SIZE
output_columns = ["lat", "lon", "cause_name", "pixel_abs_prevalence"]
reader = pd.read_csv(
    DATASET_PATH,
    chunksize=chunk_size,
    usecols=output_columns + ["metric_name"],
    dtype={
      "cause_name": "category",
    },
    compression="gzip",
)
causes: Set[str] = set()
last_size = len(causes)
i = 0
written_rows = 0
first_write = True
print(f"Start streaming with chunk size = {chunk_size}")
with gzip.open(OUTPUT_PATH, "wt", encoding="utf-8", newline="") as handle:
    while True:
      try:
        chunk = next(reader)
        filtered_chunk = chunk.loc[chunk["metric_name"] == "Percent", output_columns]
        if not filtered_chunk.empty:
          filtered_chunk.to_csv(handle, index=False, header=first_write)
          first_write = False
          written_rows += len(filtered_chunk)
        causes.update(chunk["cause_name"].cat.categories)
        cur_size = len(causes)
        if cur_size > last_size:
          print("causes = ", causes)
          last_size = cur_size
        i += 1
      except StopIteration:
        break
      except Exception as ex:
        print("#####################################################")
        print(f"Error occurred at chunk #{i}: ", ex)
        print("#####################################################")
print(f"Wrote {written_rows} rows to {OUTPUT_PATH}")
print(f"DONE after {i} chunks")

# Trying to decode the original prevalence.csv
appearently it's multiple gzipped csvs concatenated in some unknown way

In [ ]:
import gzip

In [ ]:
# sanity check if the file is valid gzip
with gzip.open(DATASET_PATH, "rb") as fin:
  try:
    while fin.read(1024 * 1024): # read 1 MB at a time:
      pass
    print("File is a valid gzip archive.")
  except Exception as e:
    print(e)

In [ ]:
# extract the first sub-file
with open(DS_CONVERTED_PATH1, "wb") as fout:
  with gzip.open(DATASET_PATH, "rb") as fin:
    try:
      while True:
        cur_chunk = fin.read(1024 * 1024) # read 1 MB at a time
        if cur_chunk:
          fout.write(cur_chunk)
        else:
          break
      print("File is a valid gzip archive.")
    except Exception as e:
      print(e)

In [ ]:
# find the start of the second sub-file
lsb = 0   # last successful byte
byte_offset = 86_948_901_888 #86_948_893_696 #86_947_921_920
chunk_size = 1 #* 1024
with gzip.open(DATASET_PATH, "rb") as fin:
  print("starting...")
  print(f"start pos = {fin.seek(byte_offset)}")
  try:
    while fin.read(chunk_size):
      lsb = fin.tell()
    print("File is a valid gzip archive.")
  except Exception as e:
    print(e)
print(f"lsb = {lsb}")

In [ ]:
# extract the second sub-file
byte_offset = 86_948_901_888
chunk_size = 1024 * 1024
with open(DS_CONVERTED_PATH2, "wb") as fout:
  with open(DATASET_PATH, "rb") as fin:
    print("starting...")
    print(f"start pos = {fin.seek(byte_offset)}")
    try:
      while True:
        cur_chunk = fin.read(chunk_size) # read 1 MB at a time
        if cur_chunk:
          fout.write(cur_chunk)
        else:
          break
      print("File is a valid gzip archive.")
    except Exception as e:
      print(e)

In [ ]:
import gzip

with gzip.open(DATASET_PATH, "rb") as f:
    i = 0
    try:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            i += 1
    except Exception as e:
        print("FAILED at MB:", i)
        print(e)

# Check filtered data

In [ ]:
# Imports
from typing import Dict, List, Optional, Set, Tuple
import gzip
import os
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Path constants
BASE_PATH = os.path.join('..', 'data')
DATASET_PATH = os.path.join(BASE_PATH, 'raw', 'physical', 'percent_filtered.csv.gz')

In [ ]:
sample = pd.read_csv(DATASET_PATH, nrows=10_000, compression="gzip")
print(sample.dtypes)
sample.head()

In [ ]:
plt.boxplot(sample['pixel_abs_prevalence'])
print(f"max = {sample['pixel_abs_prevalence'].max()}")